In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *
import re
from itertools import chain

In [0]:
def mapping(df, table_name):
    if table_name.startswith("race_events_"):
        status_map = {
            "R": "Resumed",
            "S": "Standing Start",
            "Y": "Rolling Start",
            "N": "Not Resumed"
        }
    else:
        status_map = {
            "R": "Retired",
            "D": "Disqualified",
            "E": "Excluded",
            "F": "Failed",
            "N": "Not Classified",
            "W": "Withdrawn",
            r"\N": None,
            "-": None,
            "USA": "United States",
            "UK": "United Kingdom",
            "UAE": "United Arab Emirates"
        }

    for col in df.columns:
        if df.schema[col].dataType == StringType():
            rename = F.col(col)
            for key, value in status_map.items():
                rename = F.when(F.col(col) == key, value).otherwise(rename)
            df = df.withColumn(col, rename)
    return df

def drop_url(df):
    if "url" in df.columns:
        df = df.drop("url")
    return df

def trim_strings(df):
    for col in df.columns:
        if(df.schema[col].dataType == StringType()):
            df = df.withColumn(col, F.trim(F.col(col)))
    return df

def cast_date_columns(df):
    for col in df.columns:
        if ("date" in col or "dob" in col):
            df = df.withColumn(
                col,
                F.date_format(
                    F.coalesce(
                        F.to_date(F.col(col), "yyyy-MM-dd"),
                        F.to_date(F.col(col), "MM/dd/yy"),
                    ),
                    "yyyy-MM-dd"
                )
            )
    return df

def cast_to_integer(df):
    columns = ["milliseconds", "position", "number", "grid", "rank", "year"]
    for col_name in columns:
        if col_name in [str(c) for c in df.columns]:
            df = df.withColumn(col_name, F.expr(f"try_cast(`{col_name}` AS INT)"))
    return df




# ── 3. Drop url column if it exists ──────────────────────────────────────────


# ── 4. Cast *_date columns that match yyyy-MM-dd to DateType ─────────────────


# ── 5. Cast to IntegerType (after \N nulling) ───────────────────


def transform_time_to_precision(df: DataFrame) -> DataFrame:
    # Force column names to strings to avoid Column object issues
    cols = [str(c) for c in df.columns]
    cols_lower = [c.lower() for c in cols]

    has_time_col = any("time" in c and c != "fastestlaptime" for c in cols_lower)

    if all([("duration" in cols_lower), has_time_col, ("milliseconds" in cols_lower)]):
        duration_cols = [c for c in cols if "duration" in c.lower()]
        df = df.drop(*duration_cols)
        cols = [str(c) for c in df.columns]
        cols_lower = [c.lower() for c in cols]

    elif "milliseconds" in cols_lower:
        time_cols_to_drop = [
            c for c in cols
            if "time" in c.lower()
            and c.lower() != "fastestlaptime"
            and c.lower() != "milliseconds"
        ]
        if time_cols_to_drop:
            df = df.drop(*time_cols_to_drop)
            cols = [str(c) for c in df.columns]
            cols_lower = [c.lower() for c in cols]

    targets = ["time", "duration", "q1", "q2", "q3", "fastest"]

    for col_name in cols:
        low_col = col_name.lower()

        if any(word in low_col for word in targets):
            if low_col in ["milliseconds", "fastestlap"]:
                continue

            df = df.withColumn(col_name, F.regexp_replace(F.col(col_name), r"^\+", ""))

            df = df.withColumn(
                col_name,
                F.when(~F.col(col_name).contains(":"), F.concat(F.lit("0:"), F.col(col_name)))
                 .otherwise(F.col(col_name))
            )
            df = df.withColumn(col_name, F.when(F.col(col_name).rlike(r'\\N'), None).otherwise(F.col(col_name)))
            parts = F.split(F.col(col_name), "[:.]")
            df = df.withColumn(
                col_name,
                (F.coalesce(parts.getItem(0).cast("int"), F.lit(0)) * 60000) +
                (F.coalesce(parts.getItem(1).cast("int"), F.lit(0)) * 1000) +
                (F.coalesce(parts.getItem(2).cast("int"), F.lit(0)))
            ).withColumn(col_name, F.col(col_name).cast("int"))

    return df

In [0]:
# Master function
def apply_general_silver_transforms(df, table_name):
    #df = df.replace('\\N', None)
    df = mapping(df, table_name)
    df = drop_url(df)
    df = trim_strings(df)
    df = cast_date_columns(df)
    
    #df = cast_to_integer(df)
    #df = transform_time_to_precision(df)
    return df